# Notebook 1 — Data Preparation
## Explainable Multimodal Diabetes/Metabolic Risk Framework

**Purpose:** Pull all datasets via Kaggle API, preprocess, split, and save to Google Drive.

**Datasets downloaded:**
| Module | Dataset | Kaggle Slug |
|--------|---------|-------------|
| M1 | APTOS 2019 | `aptos2019-blindness-detection` |
| M2 | HAM10000   | `kmader/skin-lesion-analysis-toward-melanoma-detection` |
| M3 | FER2013    | `msambare/fer2013` |
| M4 | NHANES     | Via `nhanes` PyPI package |
| M4 | Pima       | `uciml/pima-indians-diabetes-database` |

**Run once** — saves everything to Drive so other notebooks can load it without re-downloading.

In [ ]:
# ─── Step 0: Install dependencies ────────────────────────────────────────────
!pip install -q torch torchvision kaggle nhanes pandas numpy scikit-learn pillow tqdm

In [ ]:
# ─── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/MultimodalDisease'
os.makedirs(BASE_DIR, exist_ok=True)

# Set env var so src/config.py resolves paths to Drive
os.environ['MMDISEASE_BASE'] = BASE_DIR
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
# ─── Step 2: Clone / copy source code to Colab ───────────────────────────────
import shutil, sys

# Option A: If repo is on GitHub
# !git clone https://github.com/YOUR_USERNAME/Multimodal_Disease.git /content/Multimodal_Disease

# Option B: Copy from Drive (upload the repo zip to Drive first)
# !unzip /content/drive/MyDrive/Multimodal_Disease.zip -d /content/

# Option C: The src folder is already available in /content/Multimodal_Disease
REPO_DIR = '/content/Multimodal_Disease'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Source code path configured.')

In [ ]:
# ─── Step 3: Set up Kaggle API credentials ───────────────────────────────────
# Upload kaggle.json from: kaggle.com -> Account -> Create API Token
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)
print('Kaggle credentials configured.')

In [ ]:
# ─── Step 4: Create directory structure ──────────────────────────────────────
DATA_DIR = f'{BASE_DIR}/data'
dirs = [
    f'{DATA_DIR}/aptos2019',
    f'{DATA_DIR}/ham10000',
    f'{DATA_DIR}/fer2013',
    f'{DATA_DIR}/nhanes',
    f'{DATA_DIR}/pima',
    f'{BASE_DIR}/saved_models',
    f'{BASE_DIR}/outputs/figures/m4_shap',
    f'{BASE_DIR}/outputs/figures/m5_shap',
    f'{BASE_DIR}/outputs/scores',
    f'{BASE_DIR}/outputs/logs',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('Directory structure created.')

In [ ]:
# ─── Step 5: Download APTOS 2019 (M1 — Retinopathy) ─────────────────────────
APTOS_DIR = f'{DATA_DIR}/aptos2019'

if not os.path.exists(f'{APTOS_DIR}/train.csv'):
    !kaggle competitions download -c aptos2019-blindness-detection -p {APTOS_DIR}
    !unzip -q {APTOS_DIR}/aptos2019-blindness-detection.zip -d {APTOS_DIR}
    print('APTOS 2019 downloaded.')
else:
    print('APTOS 2019 already present — skipping.')

import pandas as pd
aptos_df = pd.read_csv(f'{APTOS_DIR}/train.csv')
print(f'APTOS: {len(aptos_df)} images')
print(aptos_df['diagnosis'].value_counts())

In [ ]:
# ─── Step 6: Download HAM10000 (M2 — Acanthosis Proxy) ──────────────────────
HAM_DIR = f'{DATA_DIR}/ham10000'

if not os.path.exists(f'{HAM_DIR}/HAM10000_metadata.csv'):
    !kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection -p {HAM_DIR}
    !unzip -q {HAM_DIR}/skin-lesion-analysis-toward-melanoma-detection.zip -d {HAM_DIR}
    # Flatten image folders
    import glob, shutil
    os.makedirs(f'{HAM_DIR}/images', exist_ok=True)
    for f in glob.glob(f'{HAM_DIR}/**/*.jpg', recursive=True):
        shutil.copy2(f, f'{HAM_DIR}/images/')
    print('HAM10000 downloaded.')
else:
    print('HAM10000 already present — skipping.')

ham_df = pd.read_csv(f'{HAM_DIR}/HAM10000_metadata.csv')
print(f'HAM10000: {len(ham_df)} samples')
print(ham_df['dx'].value_counts())

In [ ]:
# ─── Step 7: Download FER2013 (M3 — Stress) ─────────────────────────────────
FER_DIR = f'{DATA_DIR}/fer2013'

if not os.path.exists(f'{FER_DIR}/train'):
    !kaggle datasets download -d msambare/fer2013 -p {FER_DIR}
    !unzip -q {FER_DIR}/fer2013.zip -d {FER_DIR}
    print('FER2013 downloaded.')
else:
    print('FER2013 already present — skipping.')

import glob
train_imgs = glob.glob(f'{FER_DIR}/train/**/*.jpg', recursive=True)
test_imgs  = glob.glob(f'{FER_DIR}/test/**/*.jpg',  recursive=True)
print(f'FER2013: {len(train_imgs)} train, {len(test_imgs)} test images')

In [ ]:
# ─── Step 8: Download Pima Indians Diabetes (M4) ─────────────────────────────
PIMA_DIR = f'{DATA_DIR}/pima'

if not os.path.exists(f'{PIMA_DIR}/diabetes.csv'):
    !kaggle datasets download -d uciml/pima-indians-diabetes-database -p {PIMA_DIR}
    !unzip -q {PIMA_DIR}/pima-indians-diabetes-database.zip -d {PIMA_DIR}
    print('Pima dataset downloaded.')
else:
    print('Pima already present — skipping.')

pima_df = pd.read_csv(f'{PIMA_DIR}/diabetes.csv')
print(f'Pima: {pima_df.shape} | Positive rate: {pima_df["Outcome"].mean():.3f}')

In [ ]:
# ─── Step 9: Load NHANES (M4 — Primary tabular dataset) ─────────────────────
NHANES_DIR = f'{DATA_DIR}/nhanes'
nhanes_csv = f'{NHANES_DIR}/nhanes_diabetes.csv'

if not os.path.exists(nhanes_csv):
    !pip install -q nhanes
    
    # Try nhanes PyPI package
    try:
        import nhanes.load as nhanes_load
        # Load key NHANES tables from 2017-2018 cycle
        demo = nhanes_load.load_NHANES_data(
            year='2017-2018', NHANES_subset='demographics', NHANES_dataset='DEMO_J'
        )
        bmx  = nhanes_load.load_NHANES_data(
            year='2017-2018', NHANES_subset='examination', NHANES_dataset='BMX_J'
        )
        bpx  = nhanes_load.load_NHANES_data(
            year='2017-2018', NHANES_subset='examination', NHANES_dataset='BPX_J'
        )
        glu  = nhanes_load.load_NHANES_data(
            year='2017-2018', NHANES_subset='laboratory', NHANES_dataset='GLU_J'
        )
        ghb  = nhanes_load.load_NHANES_data(
            year='2017-2018', NHANES_subset='laboratory', NHANES_dataset='GHB_J'
        )
        tchol = nhanes_load.load_NHANES_data(
            year='2017-2018', NHANES_subset='laboratory', NHANES_dataset='TCHOL_J'
        )
        
        # Merge on SEQN (participant ID)
        dfs = [demo[['SEQN','RIDAGEYR','DMDEDUC2']],
               bmx[['SEQN','BMXBMI','BMXWAIST']],
               bpx[['SEQN','BPXSY1','BPXDI1']],
               glu[['SEQN','LBXGLU']],
               ghb[['SEQN','LBXGH']],
               tchol[['SEQN','LBXTC']]]
        
        from functools import reduce
        merged = reduce(lambda a, b: a.merge(b, on='SEQN', how='inner'), dfs)
        
        # Rename to standard names
        rename = {
            'RIDAGEYR': 'age', 'BMXBMI': 'bmi', 'BMXWAIST': 'waist_circumference',
            'BPXSY1': 'systolic_bp', 'BPXDI1': 'diastolic_bp',
            'LBXGLU': 'fasting_glucose', 'LBXGH': 'hba1c',
            'LBXTC': 'total_cholesterol', 'DMDEDUC2': 'education_level'
        }
        merged = merged.rename(columns=rename)
        
        # Create diabetes label (ADA criteria)
        merged['diabetes'] = ((merged['fasting_glucose'] >= 126) | 
                               (merged['hba1c'] >= 6.5)).astype(int)
        
        # Add placeholder cols for missing features
        for col in ['hdl', 'triglycerides', 'physical_activity',
                    'smoking_status', 'alcohol_use', 'family_history_diabetes']:
            if col not in merged.columns:
                merged[col] = 0  # fill with 0; note as missing in paper
        
        merged.dropna(inplace=True)
        merged.to_csv(nhanes_csv, index=False)
        print(f'NHANES: {merged.shape} | Positive rate: {merged["diabetes"].mean():.3f}')
    
    except Exception as e:
        print(f'nhanes PyPI failed ({e}) — using synthetic NHANES-like data for testing')
        # Synthetic fallback for testing the pipeline
        import numpy as np
        np.random.seed(42)
        n = 5000
        synth = pd.DataFrame({
            'age': np.random.randint(20, 80, n),
            'bmi': np.random.normal(28, 6, n).clip(15, 60),
            'waist_circumference': np.random.normal(95, 15, n).clip(60, 150),
            'systolic_bp': np.random.normal(125, 20, n).clip(80, 200),
            'diastolic_bp': np.random.normal(78, 12, n).clip(50, 120),
            'fasting_glucose': np.random.normal(105, 25, n).clip(70, 300),
            'hba1c': np.random.normal(5.8, 1.2, n).clip(4.0, 14.0),
            'total_cholesterol': np.random.normal(200, 35, n).clip(100, 400),
            'hdl': np.random.normal(52, 14, n).clip(20, 100),
            'triglycerides': np.random.normal(150, 80, n).clip(30, 800),
            'physical_activity': np.random.randint(0, 5, n),
            'smoking_status': np.random.randint(0, 3, n),
            'alcohol_use': np.random.randint(0, 5, n),
            'family_history_diabetes': np.random.randint(0, 2, n),
            'education_level': np.random.randint(1, 6, n),
        })
        synth['diabetes'] = ((synth['fasting_glucose'] >= 126) | 
                              (synth['hba1c'] >= 6.5)).astype(int)
        synth.to_csv(nhanes_csv, index=False)
        print(f'[SYNTHETIC NHANES] {synth.shape} | Positive rate: {synth["diabetes"].mean():.3f}')
        print('NOTE: Replace with real NHANES data for paper submission!')
else:
    nhanes_df = pd.read_csv(nhanes_csv)
    print(f'NHANES already present: {nhanes_df.shape} | Positive rate: {nhanes_df["diabetes"].mean():.3f}')

In [ ]:
# ─── Step 10: Dataset summary table ─────────────────────────────────────────
import glob

summary = pd.DataFrame([
    {'Module': 'M1 — Retinopathy',  'Dataset': 'APTOS 2019',
     'Samples': len(aptos_df), 'Task': '5-class severity',
     'Modality': 'Fundus image (RGB)', 'Benchmark': 'Yes (Kaggle competition)'},
    {'Module': 'M2 — Acanthosis',   'Dataset': 'HAM10000 (proxy)',
     'Samples': len(ham_df), 'Task': 'Binary (AN-like vs normal)',
     'Modality': 'Dermoscopy image (RGB)', 'Benchmark': 'Proxy (limitation noted)'},
    {'Module': 'M3 — Stress',       'Dataset': 'FER2013',
     'Samples': len(train_imgs)+len(test_imgs), 'Task': '7-class emotion → stress score',
     'Modality': 'Facial image (grayscale)', 'Benchmark': 'Yes (Kaggle)'},
    {'Module': 'M4 — Tabular',      'Dataset': 'NHANES 2017-18',
     'Samples': '~5000+', 'Task': 'Binary diabetes prediction',
     'Modality': 'Clinical/lifestyle features', 'Benchmark': 'Yes (CDC)'},
    {'Module': 'M4 — Validation',   'Dataset': 'Pima Indians',
     'Samples': len(pima_df), 'Task': 'Binary diabetes prediction',
     'Modality': 'Clinical features', 'Benchmark': 'Yes (UCI)'},
])

print('=== DATASET SUMMARY (Table 4.2 in paper) ===')
print(summary.to_string(index=False))
summary.to_csv(f'{BASE_DIR}/outputs/dataset_summary.csv', index=False)
print('\nSaved dataset summary.')

In [ ]:
# ─── Step 11: Verify data integrity ─────────────────────────────────────────
from PIL import Image
import random

def verify_images(img_dir, n=5, ext='png'):
    """Spot-check n random images can be opened."""
    files = glob.glob(f'{img_dir}/*.{ext}', recursive=False)[:100]
    sample = random.sample(files, min(n, len(files)))
    ok = 0
    for f in sample:
        try:
            Image.open(f)
            ok += 1
        except:
            print(f'  CORRUPT: {f}')
    print(f'  {ok}/{len(sample)} images OK in {img_dir}')

verify_images(f'{APTOS_DIR}/train_images', n=5, ext='png')
verify_images(f'{HAM_DIR}/images',         n=5, ext='jpg')
print('\nData integrity check passed!')
print(f'\nAll data saved to: {DATA_DIR}')
print('Ready to run Notebooks 2–5.')